# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# Explore record sets, fields, and columns in the dataset
from pprint import pprint

def list_recordsets_with_fields(ds):
    record_sets = list(ds.record_sets.values())
    if not record_sets:
        print('No record sets defined in this dataset (may be a single tabular file with columns as fields).')
    else:
        print('Available record sets:')
        for rs in record_sets:
            print(f"  - @id: {rs['@id']}  name: {rs.get('name', 'N/A')}")
            fields = rs.get('field', [])
            if isinstance(fields, dict):
                fields = [fields]
            print('    Fields:')
            for f in fields:
                if isinstance(f, str):
                    print(f'      @id: {f}')
                elif isinstance(f, dict):
                    print(f"      @id: {f.get('@id','N/A')} name: {f.get('name','N/A')}")
                else:
                    print('      (Unknown Field Definition)')

    # If there are no record sets, try printing fields directly from metadata
    if not record_sets and hasattr(ds.metadata, 'field'):
        fields = ds.metadata.field
        print('Fields in dataset:')
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            if isinstance(f, str):
                print(f'  @id: {f}')
            elif isinstance(f, dict):
                print(f"  @id: {f.get('@id','N/A')} name: {f.get('name','N/A')}")
            else:
                print('  (Unknown Field Definition)')

    # Also list columns if present
    if hasattr(ds.metadata, 'column'):
        columns = ds.metadata.column
        print('\nColumns:')
        for col in columns:
            if isinstance(col, dict):
                print(f"  @id: {col.get('@id', 'N/A')} name: {col.get('name', 'N/A')}")
            elif isinstance(col, str):
                print(f"  @id: {col}")

list_recordsets_with_fields(dataset)

## 3. Data Extraction
Load data from each record set or, if only one (or a single DataFrame), extract that, using the `@id` for referencing.

In [ ]:
# Extract all available record sets or top-level tabular data
dataframes = {}

if dataset.record_sets:
    record_sets_ids = list(dataset.record_sets.keys())
    print(f"Found record sets: {record_sets_ids}")
else:
    # Try to get a flat list of columns if record sets aren't present
    record_sets_ids = ['default']

for record_set in record_sets_ids:
    # If there are real record sets, use their @id, else just call records() with no argument
    if record_set != 'default':
        try:
            records = list(dataset.records(record_set=record_set))
        except Exception as e:
            print(f"Could not extract records for record_set {record_set}: {e}")
            continue
    else:
        try:
            records = list(dataset.records())
        except Exception as e:
            print(f"Could not extract records: {e}")
            continue
    df = pd.DataFrame(records)
    dataframes[record_set] = df

# Select the first record set/table for demonstration
main_record_set = record_sets_ids[0]
print(f"\nColumns in '{main_record_set}': {list(dataframes[main_record_set].columns)}")
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Identify numeric fields for analysis
import numpy as np

df = dataframes[main_record_set]

numeric_candidates = []
for col in df.columns:
    # Attempt to infer numeric columns by type or name
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_candidates.append(col)
    else:
        # Try conversion to numeric (ignore errors)
        try:
            df[col+'_tmp'] = pd.to_numeric(df[col], errors='coerce')
            if df[col+'_tmp'].notnull().any():
                numeric_candidates.append(col)
            df.drop(columns=[col+'_tmp'], inplace=True)
        except Exception:
            continue

print(f"Candidate numeric fields: {numeric_candidates}")

# Pick a numeric field to demonstrate filtering, normalization, and grouping
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # Use the first numeric field/column '@id'
    # Attempt to convert if not already numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = np.nanmedian(df[numeric_field])  # use median as a "median split" for demonstration
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    colnorm = f"{numeric_field}_normalized"
    filtered_df[colnorm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, colnorm]].head())

    # Select possible grouping fields (categorical)
    other_cols = [c for c in df.columns if c != numeric_field and df[c].nunique() < 20]
    group_field = other_cols[0] if other_cols else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
    else:
        print('No suitable group field found for grouping.')
else:
    print('No numeric fields found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of selected numeric field
if numeric_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=10)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If a group field was found, show boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load, inspect, and process the FAIR² dataset on second primary colorectal cancer in survivors. Exploratory analysis included data filtering, normalization, grouping, and simple visualizations. The approach shown here can be adapted to similar datasets provided in the Croissant format for robust, reproducible data science workflows.